In [0]:
from pyspark.sql.functions import col,initcap,lit

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run /Workspace/Users/chrknov6@hotmail.com/formula1/Incremental/00.Configurations

In [0]:
%run "/Workspace/Users/chrknov6@hotmail.com/formula1/Incremental/002.silver helper functions"

In [0]:
bronze_table = f"{catalog}.{bronze_schema}.races"
silver_table = f"{catalog}.{silver_schema}.races"

In [0]:
renamed_columns = {"raceName": "race_name", "circuitid": "circuit_id", "date": "race_date"}

In [0]:
races_df = (
             spark.table(bronze_table)
                  .filter(col("batch_id") == lit(v_batch_id))
                  .drop(col("url"))
                  .withColumnsRenamed(renamed_columns)
                  .dropDuplicates(["season","round"])
                  .withColumn("race_name",initcap((col("race_name"))))
)


In [0]:
write_to_silver(
    input_df=races_df,
    table_name=silver_table,
    merge_condition=((col("t.season") == col("s.season")) & (col("t.round") == col("s.round"))),
    columns_to_update= ["season","round","race_name","race_date","ingestion_time","filename"] 
    )